# Annotation Agreement and Classical Baselines

This notebook cleans and merges the human annotations, computes inter-annotator agreement, preserves the predefined train/development/test split, and evaluates majority, word TF-IDF, and character TF-IDF baselines. It is retained as research provenance from the original Colab environment, including historical Google Drive paths.


In [1]:
import pandas as pd
import numpy as np
import os
import random

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
file_path = '/content/drive/MyDrive/Interiority_26Spring/Data/annotation_updated.csv'
df_original = pd.read_csv(file_path)
df_original = df_original[["index", "title", "paragraph", "paragraph_length", "split","original_label",
         "chelsea_label_new", "michelle_label_new", #"match", "discussion",
         "final_label"]]
label_cols = [ 'original_label',
              'chelsea_label_new', 'michelle_label_new', #'match','discussion',
              'final_label']
df_original[label_cols] = df_original[label_cols].apply(lambda x: x.str.lower())
df_original['final_label'] = df_original['final_label'].str.strip()
df_original['gold_label'] = df_original['final_label'].fillna(df_original['chelsea_label_new'])
df_original

,index,title,paragraph,paragraph_length,split,original_label,chelsea_label_new,michelle_label_new,final_label,gold_label
0,0,A Room with a View,"“I have been a failure,” said Miss Bartlett, a...",50.0,train,high,NaN,NaN,high,high
1,1,A Room with a View,Lucy paused. “Cecil said one day—and I thought...,63.0,train,low,NaN,NaN,low,low
2,3,A Room with a View,An engagement is so potent a thing that sooner...,128.0,train,low,NaN,NaN,low,low
3,4,A Room with a View,“In the course of conversation they said that ...,56.0,train,low,NaN,NaN,low,low
4,7,A Room with a View,The young man named George glanced at the clev...,71.0,train,low,NaN,NaN,high,high
...,...,...,...,...,...,...,...,...,...,...
648,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
649,NaN,NaN,NaN,NaN,NaN,NaN,244,0.4114671164,NaN,244
650,NaN,NaN,NaN,NaN,NaN,NaN,199,0.3355817875,NaN,199
651,NaN,NaN,NaN,NaN,NaN,NaN,150,0.2529510961,NaN,150


In [4]:
df_original = df_original[pd.to_numeric(df_original['index'], errors='coerce').notna()]
df_original['index'] = df_original['index'].astype(int)

/tmp/ipykernel_392/2686753015.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_original['index'] = df_original['index'].astype(int)


In [5]:
df_original = df_original.dropna(subset=['index'])
df_original = df_original.sort_values(by = 'index')
df_original.reset_index(drop=True, inplace=True)
df_original.shape

(597, 10)

## Annotation cleaning and relabeling


In [6]:
df_original['gold_label'].value_counts()

,count
gold_label,
high,266
low,226
none,105


In [7]:
df_original['original_label'].value_counts()

,count
original_label,
none,237
high,204
low,156


In [8]:
match_pct = len(df_original[df_original['gold_label'] == df_original['original_label']])/len(df_original)*100
print(f"{match_pct:.1f}% of relabeled data matches original label")

60.5% of relabeled data matches original label


### Historical development/adjudication label counts

The counts below—HIGH 266, LOW 226, and NONE 105—describe the 597-passage development/adjudicated portion, not the complete benchmark. The final 717-passage dataset contains HIGH 327, LOW 274, and NONE 116. Saved numerical outputs are retained from the original execution.


## Merge additional annotations


In [9]:
file_path = '/content/drive/MyDrive/Interiority_26Spring/Data/new_annotation.csv'
df_new = pd.read_csv(file_path)
df_new['gold_label'] = df_new['final_label'].fillna(df_new['chelsea_label_new'])
print(f"Rows with NaN gold_label: {df_new['gold_label'].isnull().sum()}")
df_new

Rows with NaN gold_label: 0


,index,title,paragraph,paragraph_length,chelsea_label_new,michelle_label_new,final_label,gold_label
0,0,Emma,“But she is so fond of the nephew: he is so ve...,68,HIGH,HIGH,NaN,HIGH
1,1,Emma,“And so you actually put this piece of court-p...,65,HIGH,HIGH,NaN,HIGH
2,2,Emma,A week had not passed since Miss Hawkins’s nam...,81,LOW,LOW,NaN,LOW
3,3,Emma,It was by no means his daughter’s wish that th...,118,LOW,LOW,NaN,LOW
4,4,Emma,Emma did not repent her condescension in going...,70,HIGH,HIGH,NaN,HIGH
...,...,...,...,...,...,...,...,...
115,115,Middlemarch,At Lowick Dorothea searched desk and drawer—se...,203,LOW,LOW,NaN,LOW
116,116,Middlemarch,Her mind was evidently arrested by some sudden...,78,LOW,LOW,NaN,LOW
117,117,Middlemarch,"And so they remained for many minutes, opposit...",83,HIGH,HIGH,NaN,HIGH
118,118,Middlemarch,"Mr. Bulstrode was a member of the Board, and j...",102,LOW,LOW,NaN,LOW


In [10]:
from sklearn.metrics import cohen_kappa_score

# Make sure labels are in the intended ordinal order
label_order = ["NONE", "LOW", "HIGH"]

y1 = df_new["chelsea_label_new"]
y2 = df_new["michelle_label_new"]

# 1. Unweighted Cohen's kappa
kappa = cohen_kappa_score(
    y1,
    y2,
    labels=label_order
)

print(f"Cohen's kappa: {kappa:.4f}")

# 2. Linear weighted Cohen's kappa
linear_weighted_kappa = cohen_kappa_score(
    y1,
    y2,
    labels=label_order,
    weights="linear"
)

print(f"Linear weighted Cohen's kappa: {linear_weighted_kappa:.4f}")

# 3. Quadratic weighted Cohen's kappa
quadratic_weighted_kappa = cohen_kappa_score(
    y1,
    y2,
    labels=label_order,
    weights="quadratic"
)

print(f"Quadratic weighted Cohen's kappa: {quadratic_weighted_kappa:.4f}")

Cohen's kappa: 0.6409
Linear weighted Cohen's kappa: 0.6903
Quadratic weighted Cohen's kappa: 0.7572


In [11]:
!pip install krippendorff

In [12]:
import numpy as np
import krippendorff

# Map ordinal labels to numeric values
label_mapping = {
    "NONE": 0,
    "LOW": 1,
    "HIGH": 2
}

y1_num = df_new["chelsea_label_new"].map(label_mapping).to_numpy()
y2_num = df_new["michelle_label_new"].map(label_mapping).to_numpy()

# Krippendorff expects shape: (annotators, items)
reliability_data = np.vstack([y1_num, y2_num])

# 1. Nominal Krippendorff's alpha
alpha_nominal = krippendorff.alpha(
    reliability_data=reliability_data,
    level_of_measurement="nominal"
)

print(f"Krippendorff's alpha (nominal): {alpha_nominal:.4f}")

# 2. Ordinal Krippendorff's alpha
alpha_ordinal = krippendorff.alpha(
    reliability_data=reliability_data,
    level_of_measurement="ordinal"
)

print(f"Krippendorff's alpha (ordinal): {alpha_ordinal:.4f}")

Krippendorff's alpha (nominal): 0.6424
Krippendorff's alpha (ordinal): 0.7626


In [13]:
# Merge the adjudicated and additional annotations.
df_merged = pd.concat([df_original, df_new], ignore_index=True).copy()
#df_merged['index'] = range(len(df_merged))
#df = df.drop(columns=['index'])

df_merged
#df_merged.to_csv('merged.csv', index=False)

,index,title,paragraph,paragraph_length,split,original_label,chelsea_label_new,michelle_label_new,final_label,gold_label
0,0,A Room with a View,"“I have been a failure,” said Miss Bartlett, a...",50.0,train,high,NaN,NaN,high,high
1,1,A Room with a View,Lucy paused. “Cecil said one day—and I thought...,63.0,train,low,NaN,NaN,low,low
2,2,A Room with a View,"Miss Bartlett, who was poor at figures, became...",NaN,train,high,low,low,low,low
3,3,A Room with a View,An engagement is so potent a thing that sooner...,128.0,train,low,NaN,NaN,low,low
4,4,A Room with a View,“In the course of conversation they said that ...,56.0,train,low,NaN,NaN,low,low
...,...,...,...,...,...,...,...,...,...,...
712,115,Middlemarch,At Lowick Dorothea searched desk and drawer—se...,203.0,NaN,NaN,LOW,LOW,NaN,LOW
713,116,Middlemarch,Her mind was evidently arrested by some sudden...,78.0,NaN,NaN,LOW,LOW,NaN,LOW
714,117,Middlemarch,"And so they remained for many minutes, opposit...",83.0,NaN,NaN,HIGH,HIGH,NaN,HIGH
715,118,Middlemarch,"Mr. Bulstrode was a member of the Board, and j...",102.0,NaN,NaN,LOW,LOW,NaN,LOW


In [14]:
# Update the predefined split for the additional books.

df_merged.loc[df_merged['title'] == 'Emma', 'split'] = 'train'
df_merged.loc[df_merged['title'] == 'Middlemarch', 'split'] = 'test'

DEV_BOOK = ["The Great Gatsby", "The Garden Party, and Other Stories",'Jane Eyre: An Autobiography']
df_merged.loc[df_merged["title"].isin(DEV_BOOK), "split"] = "dev"

In [15]:
df_merged['gold_label'] = df_merged['final_label'].fillna(df_merged['chelsea_label_new'])
print(f"Rows with NaN gold_label: {df_merged['gold_label'].isnull().sum()}")

Rows with NaN gold_label: 0


In [16]:
df = df_merged[["title", "paragraph", "paragraph_length", "split", #"original_label",
        # "chelsea_label_new", "michelle_label_new", #"match", "discussion","final_label",
                'gold_label']].copy()
#label_cols = [ 'original_label', 'chelsea_label_new', 'michelle_label_new', #'match','discussion', 'final_label']
#df['final_label'] = df['final_label'].str.lower()
df['gold_label'] = df['gold_label'].str.lower()
df

,title,paragraph,paragraph_length,split,gold_label
0,A Room with a View,"“I have been a failure,” said Miss Bartlett, a...",50.0,train,high
1,A Room with a View,Lucy paused. “Cecil said one day—and I thought...,63.0,train,low
2,A Room with a View,"Miss Bartlett, who was poor at figures, became...",NaN,train,low
3,A Room with a View,An engagement is so potent a thing that sooner...,128.0,train,low
4,A Room with a View,“In the course of conversation they said that ...,56.0,train,low
...,...,...,...,...,...
712,Middlemarch,At Lowick Dorothea searched desk and drawer—se...,203.0,test,low
713,Middlemarch,Her mind was evidently arrested by some sudden...,78.0,test,low
714,Middlemarch,"And so they remained for many minutes, opposit...",83.0,test,high
715,Middlemarch,"Mr. Bulstrode was a member of the Board, and j...",102.0,test,low


In [17]:
label_map = {
    "none": 0,
    "low": 1,
    "high": 2
}

df["label_id"] = df["gold_label"].map(label_map)
df[["gold_label","label_id"]].head()


,gold_label,label_id
0,high,2
1,low,1
2,low,1
3,low,1
4,low,1


In [18]:
#df.to_csv('df_annotation_final.csv', index=False)

In [19]:
df['label_id'].value_counts()

,count
label_id,
2,327
1,274
0,116


In [20]:
pd.crosstab(df["title"], df["gold_label"], margins=True)

gold_label,high,low,none,All
title,,,,
A Room with a View,19,19,1,39
A farewell to arms,7,22,11,40
Dubliners,12,16,12,40
Emma,18,18,4,40
Jane Eyre: An Autobiography,20,17,3,40
Martin Eden,23,12,5,40
Metamorphosis,18,15,6,39
Middlemarch,23,13,4,40
Mrs. Dalloway,31,6,3,40


In [21]:
df.groupby("title")["label_id"].mean().sort_values(ascending=False)

,label_id
title,
Mrs. Dalloway,1.700000
Swann's Way,1.625000
The Picture of Dorian Gray,1.600000
Middlemarch,1.475000
A Room with a View,1.461538
Martin Eden,1.450000
Jane Eyre: An Autobiography,1.425000
The murder of Roger Ackroyd,1.350000
Emma,1.350000


In [22]:
df.groupby("title")["paragraph_length"].mean().sort_values(ascending=False)

,paragraph_length
title,
Metamorphosis,245.833333
Swann's Way,187.760000
The Dunwich horror,154.352941
My Ántonia,138.034483
Dubliners,133.280000
Martin Eden,133.142857
A farewell to arms,132.208333
Middlemarch,120.625000
The Picture of Dorian Gray,119.766667


In [23]:
df["paragraph_length"].mean()

np.float64(124.18087318087318)

In [24]:
df.head()

,title,paragraph,paragraph_length,split,gold_label,label_id
0,A Room with a View,"“I have been a failure,” said Miss Bartlett, a...",50.0,train,high,2
1,A Room with a View,Lucy paused. “Cecil said one day—and I thought...,63.0,train,low,1
2,A Room with a View,"Miss Bartlett, who was poor at figures, became...",NaN,train,low,1
3,A Room with a View,An engagement is so potent a thing that sooner...,128.0,train,low,1
4,A Room with a View,“In the course of conversation they said that ...,56.0,train,low,1


## Train, development, and test split


In [25]:
#DEV_BOOK = ["The Great Gatsby", "The Garden Party, and Other Stories"]
#df.loc[df["title"].isin(DEV_BOOK), "split"] = "dev"

train_df = df[df["split"] == "train"].reset_index(drop=True)
test_df    = df[df["split"] == "test"].reset_index(drop=True)
dev_df    = df[df["split"] == "dev"].reset_index(drop=True)

#dev_df   = train_full[train_full["title"] == DEV_BOOK].reset_index(drop=True)
#train_df = train_full[train_full["title"] != DEV_BOOK].reset_index(drop=True)

print(f"Train: {len(train_df)}  (books: {train_df['title'].nunique()})")
print(f"Dev:   {len(dev_df)}  (book: {DEV_BOOK})")
print(f"Test:  {len(test_df)}  (books: {test_df['title'].nunique()})")

print("\nTrain label dist:")
print(train_df["gold_label"].value_counts())
print("\nDev label dist:")
print(dev_df["gold_label"].value_counts())

Train: 357  (books: 9)
Dev:   120  (book: ['The Great Gatsby', 'The Garden Party, and Other Stories', 'Jane Eyre: An Autobiography'])
Test:  240  (books: 6)

Train label dist:
gold_label
high    170
low     131
none     56
Name: count, dtype: int64

Dev label dist:
gold_label
high    53
low     50
none    17
Name: count, dtype: int64


## Classical baselines

The following sections evaluate the majority-class baseline, word TF-IDF with logistic regression, and character TF-IDF with logistic regression using the predefined dataset split.


In [26]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score, f1_score, classification_report, confusion_matrix
)
from sklearn.dummy import DummyClassifier

In [27]:
train_full = pd.concat([train_df, dev_df]).reset_index(drop=True)
#train_df = df[df["split"] == "train"].reset_index(drop=True)
#test_df  = df[df["split"] == "test"].reset_index(drop=True)
print(f"\nTrain size: {len(train_full)}")
print(f"Test size:  {len(test_df)}")
print(f"\nTrain label distribution:\n{train_df['gold_label'].value_counts()}")
print(f"\nTest label distribution:\n{test_df['gold_label'].value_counts()}")


Train size: 477
Test size:  240

Train label distribution:
gold_label
high    170
low     131
none     56
Name: count, dtype: int64

Test label distribution:
gold_label
high    104
low      93
none     43
Name: count, dtype: int64


In [28]:
print(f"Training data: {train_df['title'].unique()}")
print(f"Test data: {test_df['title'].unique()}")

Training data: ['A Room with a View' 'Martin Eden' 'Metamorphosis' 'Mrs. Dalloway'
 "Swann's Way" 'The Age of Innocence'
 'The Awakening, and Selected Short Stories' 'The Dunwich horror' 'Emma']
Test data: ['A farewell to arms' 'Dubliners' 'My Ántonia'
 'The Picture of Dorian Gray' 'The murder of Roger Ackroyd' 'Middlemarch']


In [29]:
X_train, y_train = train_df["paragraph"].values, train_df["label_id"].values
X_test,  y_test  = test_df["paragraph"].values,  test_df["label_id"].values

label_names = ["none", "low", "high"]

# Compute the shared evaluation metrics.
def evaluate(name, y_true, y_pred):
    acc = accuracy_score(y_true, y_pred)
    macro_f1 = f1_score(y_true, y_pred, average="macro")
    print(f"\n=== {name} ===")
    print(f"Accuracy : {acc:.4f}")
    print(f"Macro-F1 : {macro_f1:.4f}")
    print("\nPer-class report:")
    print(classification_report(y_true, y_pred, target_names=label_names, digits=4))
    print("Confusion matrix (rows=true, cols=pred):")
    cm = confusion_matrix(y_true, y_pred)
    cm_df = pd.DataFrame(cm, index=label_names, columns=label_names)
    print(cm_df)
    return {"name": name, "accuracy": acc, "macro_f1": macro_f1}

results = []

In [30]:
# Baseline 1: majority class
dummy = DummyClassifier(strategy="most_frequent", random_state=42)
dummy.fit(X_train, y_train)
results.append(evaluate("Majority Class", y_test, dummy.predict(X_test)))


=== Majority Class ===
Accuracy : 0.4333
Macro-F1 : 0.2016

Per-class report:
              precision    recall  f1-score   support

        none     0.0000    0.0000    0.0000        43
         low     0.0000    0.0000    0.0000        93
        high     0.4333    1.0000    0.6047       104

    accuracy                         0.4333       240
   macro avg     0.1444    0.3333    0.2016       240
weighted avg     0.1878    0.4333    0.2620       240

Confusion matrix (rows=true, cols=pred):
      none  low  high
none     0    0    43
low      0    0    93
high     0    0   104


/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


In [31]:
# Baseline 2: word TF-IDF with logistic regression
# Use word unigrams and bigrams.
tfidf = TfidfVectorizer(
    ngram_range=(1, 2),
    min_df=2,           # Require terms to occur in at least two documents.
    max_df=0.95,
    sublinear_tf=True,  # Apply sublinear term-frequency scaling.
    strip_accents="unicode",
    lowercase=True,
)
X_train_tfidf = tfidf.fit_transform(X_train)
X_test_tfidf  = tfidf.transform(X_test)
print(f"\nTF-IDF vocabulary size: {len(tfidf.vocabulary_)}")

lr = LogisticRegression(
    max_iter=2000,
    C=1.0,
    class_weight=None,
    random_state=42,
    n_jobs=-1,
)
lr.fit(X_train_tfidf, y_train)
results.append(evaluate("TF-IDF + LogReg", y_test, lr.predict(X_test_tfidf)))


TF-IDF vocabulary size: 7626

=== TF-IDF + LogReg ===
Accuracy : 0.5250
Macro-F1 : 0.3560

Per-class report:
              precision    recall  f1-score   support

        none     0.0000    0.0000    0.0000        43
         low     0.4545    0.3226    0.3774        93
        high     0.5517    0.9231    0.6906       104

    accuracy                         0.5250       240
   macro avg     0.3354    0.4152    0.3560       240
weighted avg     0.4152    0.5250    0.4455       240

Confusion matrix (rows=true, cols=pred):
      none  low  high
none     0   28    15
low      0   30    63
high     0    8    96


/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


In [44]:
y_pred_lr = lr.predict(X_test_tfidf)

ci = bootstrap_macro_f1(y_test, y_pred_lr,
                        groups=test_df["title"],        # Use test-book identifiers as bootstrap clusters.
                        B=2000, method="bca")
print(f"Macro-F1 = {ci['macro_f1']:.3f} "
      f"[{ci['ci_lo']:.3f}, {ci['ci_hi']:.3f}] (95% BCa)")

Macro-F1 = 0.356 [0.307, 0.427] (95% BCa)


In [33]:
# Baseline 3: character TF-IDF with logistic regression
tfidf_char = TfidfVectorizer(
    analyzer="char_wb",
    ngram_range=(3, 5),
    min_df=2,
    max_df=0.95,
    sublinear_tf=True,
)
X_train_char = tfidf_char.fit_transform(X_train)
X_test_char  = tfidf_char.transform(X_test)

lr_char = LogisticRegression(max_iter=2000, C=1.0, random_state=42, n_jobs=-1)
lr_char.fit(X_train_char, y_train)
results.append(evaluate("TF-IDF (char 3-5) + LogReg", y_test, lr_char.predict(X_test_char)))


=== TF-IDF (char 3-5) + LogReg ===
Accuracy : 0.5958
Macro-F1 : 0.4231

Per-class report:
              precision    recall  f1-score   support

        none     0.0000    0.0000    0.0000        43
         low     0.5402    0.5054    0.5222        93
        high     0.6275    0.9231    0.7471       104

    accuracy                         0.5958       240
   macro avg     0.3892    0.4762    0.4231       240
weighted avg     0.4812    0.5958    0.5261       240

Confusion matrix (rows=true, cols=pred):
      none  low  high
none     0   32    11
low      0   47    46
high     0    8    96


/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


In [36]:
y_pred_lr = lr_char.predict(X_test_char)

ci = bootstrap_macro_f1(y_test, y_pred_lr,
                        groups=test_df["title"],
                        B=2000,
                        method="percentile")
print(f"Macro-F1 = {ci['macro_f1']:.3f} "
      f"[{ci['ci_lo']:.3f}, {ci['ci_hi']:.3f}] (95% BCa)")

NameError: name 'bootstrap_macro_f1' is not defined

In [39]:
#print("\n" + "="*50)
print("SUMMARY")
print(pd.DataFrame(results).to_string(index=False))

SUMMARY
                      name  accuracy  macro_f1
            Majority Class  0.433333  0.201550
           TF-IDF + LogReg  0.525000  0.356002
TF-IDF (char 3-5) + LogReg  0.595833  0.423101


In [38]:
test_df_eval = test_df.copy()
test_df_eval["pred"] = lr_char.predict(X_test_char)
for book, group in test_df_eval.groupby("title"):
    f1 = f1_score(group["label_id"], group["pred"], average="macro", zero_division=0)
    print(f"{book:40s}  n={len(group):3d}  macro-F1={f1:.3f}")

A farewell to arms                        n= 40  macro-F1=0.399
Dubliners                                 n= 40  macro-F1=0.431
Middlemarch                               n= 40  macro-F1=0.406
My Ántonia                                n= 40  macro-F1=0.341
The Picture of Dorian Gray                n= 40  macro-F1=0.515
The murder of Roger Ackroyd               n= 40  macro-F1=0.347


In [ ]:
results

In [45]:
# Bootstrap report for the final test-set results

BOOTSTRAP_N = 1000
BOOTSTRAP_CONFIDENCE = 0.95
BOOTSTRAP_SEED = 2026
BOOTSTRAP_LABELS = ['HIGH', 'LOW', 'NONE']


def _as_label_array(values):
    return np.array([str(v).strip().upper() if str(v).strip().upper() in BOOTSTRAP_LABELS else 'INVALID' for v in values])


def _f1_for_label(labels, preds, target_label):
    labels = _as_label_array(labels)
    preds = _as_label_array(preds)
    tp = np.sum((labels == target_label) & (preds == target_label))
    fp = np.sum((labels != target_label) & (preds == target_label))
    fn = np.sum((labels == target_label) & (preds != target_label))
    denom = (2 * tp) + fp + fn
    return float((2 * tp) / denom) if denom else 0.0


def metric_accuracy(labels, preds):
    labels = _as_label_array(labels)
    preds = _as_label_array(preds)
    return float(np.mean(labels == preds)) if len(labels) else np.nan


def metric_micro_f1(labels, preds):
    labels = _as_label_array(labels)
    preds = _as_label_array(preds)
    total_tp = total_fp = total_fn = 0
    for label in BOOTSTRAP_LABELS:
        total_tp += np.sum((labels == label) & (preds == label))
        total_fp += np.sum((labels != label) & (preds == label))
        total_fn += np.sum((labels == label) & (preds != label))
    denom = (2 * total_tp) + total_fp + total_fn
    return float((2 * total_tp) / denom) if denom else 0.0


def metric_macro_f1(labels, preds):
    return float(np.mean([_f1_for_label(labels, preds, label) for label in BOOTSTRAP_LABELS]))


def metric_weighted_f1(labels, preds):
    labels_arr = _as_label_array(labels)
    supports = np.array([np.sum(labels_arr == label) for label in BOOTSTRAP_LABELS], dtype=float)
    f1s = np.array([_f1_for_label(labels, preds, label) for label in BOOTSTRAP_LABELS], dtype=float)
    total_support = supports.sum()
    return float(np.sum(f1s * supports) / total_support) if total_support else 0.0


def make_label_f1_metric(target_label):
    return lambda labels, preds: _f1_for_label(labels, preds, target_label)


def bootstrap_ci(labels, preds, metric_fn, n_bootstrap=1000, confidence=0.95, seed=42):
    """Bootstrap point estimate and percentile confidence interval.

    labels and preds are resampled as paired observations so each prediction
    remains matched to its original gold label.
    """
    labels = np.array(labels)
    preds = np.array(preds)
    n = len(labels)
    if n == 0:
        return np.nan, np.nan, np.nan

    rng = np.random.default_rng(seed)
    boot_scores = []
    for _ in range(n_bootstrap):
        sample_idx = rng.integers(0, n, size=n)
        boot_scores.append(metric_fn(labels[sample_idx], preds[sample_idx]))

    alpha = (1 - confidence) / 2
    lower = np.percentile(boot_scores, 100 * alpha)
    upper = np.percentile(boot_scores, 100 * (1 - alpha))
    point = metric_fn(labels, preds)
    return float(point), float(lower), float(upper)


BOOTSTRAP_METRICS = {
    'accuracy': metric_accuracy,
    'micro_f1': metric_micro_f1,
    'macro_f1': metric_macro_f1,
    'weighted_f1': metric_weighted_f1,
    'high_f1': make_label_f1_metric('HIGH'),
    'low_f1': make_label_f1_metric('LOW'),
    'none_f1': make_label_f1_metric('NONE'),
}

bootstrap_rows = []
for name, model_df in pd.DataFrame(results).groupby('name', sort=False):
    labels = model_df['gold_label'].to_numpy()
    preds = model_df['parsed_label'].to_numpy()
    n_items = len(model_df)
    invalid_output_count = int(np.sum(_as_label_array(preds) == 'INVALID'))

    for metric_name, metric_fn in BOOTSTRAP_METRICS.items():
        point, lower, upper = bootstrap_ci(
            labels,
            preds,
            metric_fn,
            n_bootstrap=BOOTSTRAP_N,
            confidence=BOOTSTRAP_CONFIDENCE,
            seed=BOOTSTRAP_SEED,
        )
        bootstrap_rows.append({
            'model_name': model_name,
            'metric': metric_name,
            'point': point,
            'ci_lower': lower,
            'ci_upper': upper,
            'ci_width': upper - lower,
            'n_items': n_items,
            'invalid_output_count': invalid_output_count,
            'n_bootstrap': BOOTSTRAP_N,
            'confidence': BOOTSTRAP_CONFIDENCE,
        })

final_test_bootstrap_report = pd.DataFrame(bootstrap_rows)

display(
    final_test_bootstrap_report.assign(
        estimate_ci=final_test_bootstrap_report.apply(
            lambda r: f"{r['point']:.3f} [{r['ci_lower']:.3f}, {r['ci_upper']:.3f}]",
            axis=1,
        )
    ).pivot(index='model_name', columns='metric', values='estimate_ci')
)


KeyError: 'gold_label'

In [41]:
import numpy as np
from sklearn.metrics import f1_score
from scipy.stats import norm

def _macro_f1(yt, yp, labels):
    return f1_score(yt, yp, average="macro", labels=labels, zero_division=0)

def bootstrap_macro_f1(y_true, y_pred, groups=None, B=2000, alpha=0.05,
                       seed=42, method="bca"):
    """
    Bootstrap CI for macro-F1. Only y_pred is fixed (model 不重训),
    因此 CI 反映的是 test set sampling variability, 不含 training variability.
    groups: array-like, e.g. book_id —— 有聚类结构时按 group 重抽样.
    """
    y_true, y_pred = np.asarray(y_true), np.asarray(y_pred)
    labels = np.unique(y_true)                     # Keep the label set fixed across bootstrap samples.
    theta_hat = _macro_f1(y_true, y_pred, labels)

    if groups is None:
        unit_idx = [np.array([i]) for i in range(len(y_true))]
    else:
        groups = np.asarray(groups)
        unit_idx = [np.where(groups == g)[0] for g in np.unique(groups)]
    n = len(unit_idx)
    rng = np.random.default_rng(seed)

    boot = np.empty(B)
    for b in range(B):
        sel = rng.integers(0, n, n)
        idx = np.concatenate([unit_idx[i] for i in sel])
        boot[b] = _macro_f1(y_true[idx], y_pred[idx], labels)

    if method == "percentile":
        lo, hi = np.percentile(boot, [100*alpha/2, 100*(1-alpha/2)])
    elif method == "bca":
        # bias-correction
        z0 = norm.ppf(np.mean(boot < theta_hat))
        # acceleration via jackknife over units
        jack = np.array([
            _macro_f1(*[arr[np.concatenate(unit_idx[:i] + unit_idx[i+1:])]
                        for arr in (y_true, y_pred)], labels)
            for i in range(n)
        ])
        d = jack.mean() - jack
        a = (d**3).sum() / (6.0 * (d**2).sum()**1.5 + 1e-12)
        zl, zu = norm.ppf(alpha/2), norm.ppf(1-alpha/2)
        al = norm.cdf(z0 + (z0+zl)/(1-a*(z0+zl)))
        au = norm.cdf(z0 + (z0+zu)/(1-a*(z0+zu)))
        lo, hi = np.percentile(boot, [100*al, 100*au])
    else:
        raise ValueError(method)
    return {"macro_f1": theta_hat, "ci_lo": lo, "ci_hi": hi,
            "se": boot.std(ddof=1), "boot": boot}